# BERTScore 测试 & 模型下载

测试BERTScore的性能,预下载roberta-large模型

## 1. 检查bert-score是否安装

In [ ]:
# 检查bert-score版本
!pip show bert-score

## 2. 导入并下载模型

首次运行会下载roberta-large模型(~1.4GB)

In [ ]:
from bert_score import score as bert_score
import time

print("✅ BERTScore imported successfully")
print("📥 Now downloading roberta-large model (this may take a few minutes)...")

## 3. 单次计算测试

In [ ]:
# 测试单个QA pair
reference = "Caroline went to the LGBTQ support group on 7 May 2023"
candidate = "On 7 May 2023, Caroline attended an LGBTQ support group meeting"

print("🧪 Testing single BERTScore calculation...")
print(f"Reference: {reference}")
print(f"Candidate: {candidate}")
print()

start_time = time.time()

# 使用和MemOS相同的参数
P, R, F1 = bert_score(
    [candidate],
    [reference],
    lang='en',  # 使用roberta-large
    rescale_with_baseline=True,  # MemOS的配置
    verbose=True  # 显示详细信息
)

elapsed = time.time() - start_time

print(f"\n⏱️  Time taken: {elapsed:.2f} seconds")
print(f"📊 Precision: {P.item():.4f}")
print(f"📊 Recall: {R.item():.4f}")
print(f"📊 F1: {F1.item():.4f}")

## 4. 批量计算测试

测试批量计算vs逐个计算的性能差异

In [ ]:
# 准备测试数据
test_pairs = [
    {
        "reference": "Caroline went to the LGBTQ support group on 7 May 2023",
        "candidate": "On 7 May 2023, Caroline attended an LGBTQ support group meeting"
    },
    {
        "reference": "Melanie painted a sunrise in 2022",
        "candidate": "In 2022, Melanie created a sunrise painting"
    },
    {
        "reference": "Caroline researched adoption agencies",
        "candidate": "Caroline looked into various adoption agencies"
    },
    {
        "reference": "The Sunday before 25 May 2023",
        "candidate": "Sunday, May 21, 2023"
    },
    {
        "reference": "Transgender woman",
        "candidate": "She is transgender"
    },
]

print(f"📝 Testing {len(test_pairs)} QA pairs\n")

### 4.1 逐个计算 (慢)

In [ ]:
print("🐌 Method 1: Individual calculation (one-by-one)")

start_time = time.time()
individual_scores = []

for i, pair in enumerate(test_pairs, 1):
    _, _, f1 = bert_score(
        [pair['candidate']],
        [pair['reference']],
        lang='en',
        rescale_with_baseline=True,
        verbose=False
    )
    individual_scores.append(f1.item())
    print(f"  QA {i}/5: F1 = {f1.item():.4f}")

individual_time = time.time() - start_time
print(f"\n⏱️  Total time: {individual_time:.2f} seconds")
print(f"⏱️  Average per QA: {individual_time/len(test_pairs):.2f} seconds")

### 4.2 批量计算 (快)

In [ ]:
print("🚀 Method 2: Batch calculation (all at once)")

# 准备批量数据
all_candidates = [pair['candidate'] for pair in test_pairs]
all_references = [pair['reference'] for pair in test_pairs]

start_time = time.time()

# 一次性计算所有
P, R, F1 = bert_score(
    all_candidates,
    all_references,
    lang='en',
    rescale_with_baseline=True,
    verbose=False
)

batch_time = time.time() - start_time
batch_scores = F1.tolist()

for i, score in enumerate(batch_scores, 1):
    print(f"  QA {i}/5: F1 = {score:.4f}")

print(f"\n⏱️  Total time: {batch_time:.2f} seconds")
print(f"⏱️  Average per QA: {batch_time/len(test_pairs):.2f} seconds")
print(f"\n🎯 Speedup: {individual_time/batch_time:.2f}x faster!")

## 5. 大规模测试 (模拟150个QA pairs)

预估sample版本测试的时间

In [ ]:
# 模拟150个QA pairs
num_qa = 150

# 复制测试数据
large_candidates = [pair['candidate'] for pair in test_pairs] * (num_qa // len(test_pairs) + 1)
large_references = [pair['reference'] for pair in test_pairs] * (num_qa // len(test_pairs) + 1)

large_candidates = large_candidates[:num_qa]
large_references = large_references[:num_qa]

print(f"🧪 Testing {num_qa} QA pairs in batch mode...")

start_time = time.time()

P, R, F1 = bert_score(
    large_candidates,
    large_references,
    lang='en',
    rescale_with_baseline=True,
    verbose=False
)

large_batch_time = time.time() - start_time

print(f"\n⏱️  Total time: {large_batch_time:.2f} seconds ({large_batch_time/60:.1f} minutes)")
print(f"⏱️  Average per QA: {large_batch_time/num_qa:.2f} seconds")
print(f"\n📊 Mean F1: {F1.mean().item():.4f}")
print(f"📊 Std F1: {F1.std().item():.4f}")

# 预估完整测试时间
total_qa_sample = 150  # sample版本
total_qa_full = 1986   # full版本

estimated_sample = (large_batch_time / num_qa) * total_qa_sample
estimated_full = (large_batch_time / num_qa) * total_qa_full

print(f"\n🔮 Estimated BERTScore time for LoCoMo tests:")
print(f"   Sample test (150 QA): {estimated_sample/60:.1f} minutes")
print(f"   Full test (1986 QA): {estimated_full/60:.1f} minutes ({estimated_full/3600:.1f} hours)")

## 6. 检查模型缓存位置

In [ ]:
import os
from pathlib import Path

# BERTScore缓存位置
cache_dir = Path.home() / '.cache' / 'huggingface'

print(f"📦 Hugging Face cache directory: {cache_dir}")
print(f"📦 Exists: {cache_dir.exists()}")

if cache_dir.exists():
    # 查找roberta模型
    roberta_dirs = list(cache_dir.glob('**/roberta*'))
    if roberta_dirs:
        print(f"\n✅ Found {len(roberta_dirs)} roberta model files/dirs")
        for d in roberta_dirs[:5]:  # 只显示前5个
            print(f"   - {d.name}")
    else:
        print("\n⚠️  No roberta models found in cache")

# 检查磁盘空间
!df -h ~/

## 7. GPU加速测试 (如果有GPU)

In [ ]:
import torch

print(f"🖥️  PyTorch version: {torch.__version__}")
print(f"🖥️  CUDA available: {torch.cuda.is_available()}")
print(f"🖥️  MPS (Metal) available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

if torch.cuda.is_available():
    print(f"🖥️  CUDA device: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"🖥️  Using Apple Silicon GPU (MPS)")
    device = 'mps'
else:
    print(f"🖥️  Using CPU")
    device = 'cpu'

print(f"\n🎯 Selected device: {device}")

# 测试GPU加速
if device != 'cpu':
    print(f"\n🚀 Testing with {device.upper()} acceleration...")
    
    start_time = time.time()
    
    P, R, F1 = bert_score(
        all_candidates[:10],
        all_references[:10],
        lang='en',
        rescale_with_baseline=True,
        device=device,
        verbose=False
    )
    
    gpu_time = time.time() - start_time
    print(f"⏱️  {device.upper()} time (10 QA): {gpu_time:.2f} seconds")
    
    # 比较CPU时间
    start_time = time.time()
    
    P, R, F1 = bert_score(
        all_candidates[:10],
        all_references[:10],
        lang='en',
        rescale_with_baseline=True,
        device='cpu',
        verbose=False
    )
    
    cpu_time = time.time() - start_time
    print(f"⏱️  CPU time (10 QA): {cpu_time:.2f} seconds")
    print(f"\n🎯 {device.upper()} speedup: {cpu_time/gpu_time:.2f}x faster!")
else:
    print("\n⚠️  No GPU available, skipping GPU test")

## 总结

运行完这个notebook后,你会知道:
1. ✅ roberta-large模型已下载
2. ✅ 单次计算耗时
3. ✅ 批量计算的加速比
4. ✅ 预估的总测试时间
5. ✅ 是否可以使用GPU加速

**建议**: 如果批量计算够快(150个QA < 5分钟),就保留BERTScore;否则考虑跳过或只在最终结果时计算。